Vamos ahora a tratar las imputaciones registradas con el sistema MES

En las imputaciones es donde esta el detalle real de ejecución (qué máquina, qué tiempos reales, quién y cuándo).

In [17]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import importlib
import pickle

In [18]:
importlib.reload(pd)

<module 'pandas' from 'c:\\Users\\jaume\\Documents\\01 Documentos\\01 Estudios\\IA\\TFM\\TFM\\.venv_tfm\\Lib\\site-packages\\pandas\\__init__.py'>

In [19]:
ruta_imputations = os.path.join("res\\raw", "imputaciones.csv")
print(ruta_imputations)

res\raw\imputaciones.csv


In [20]:
columns_imputations = ["FechaCreacionDte",	"smNodoId",	"smNodoNme",	"smOrdenId",	"smMaterialId",	"smOrdenNme",	"smFaseId",	"smFaseNme",	"CantidadManualOKNbr",	"CantidadManualNOKNbr",	"MinutosRealNbr",	"MinutosPreparacionNbr",	"MinutosProduccionNbr",	"PorcentajeDedicacionNbr",	"ProductivoInd",	"smIncidenciaId",	"smIncidenciaNme",	"smFasePersonalizado1Txt",	"smNodoId2",	"smNodoPersonalizado1Nbr",	"smNodoPersonalizado2Nbr",	"ssEnumFaseEstadoId",	"SmFaseUid",	"TraspasadoInd",	"JsonBodyTxt",	"JsonResponseTxt"]

pd.set_option("display.max_columns", None)  
pd.set_option("display.width", None)        

df_imputations = pd.read_csv(ruta_imputations, sep=";", header=None, names=columns_imputations)

C:\Users\jaume\AppData\Local\Temp\ipykernel_14316\1223014052.py:6: DtypeWarning: Columns (6,15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df_imputations = pd.read_csv(ruta_imputations, sep=";", header=None, names=columns_imputations)


In [21]:
display(df_imputations.sample(1))

,FechaCreacionDte,smNodoId,smNodoNme,smOrdenId,smMaterialId,smOrdenNme,smFaseId,smFaseNme,CantidadManualOKNbr,CantidadManualNOKNbr,MinutosRealNbr,MinutosPreparacionNbr,MinutosProduccionNbr,PorcentajeDedicacionNbr,ProductivoInd,smIncidenciaId,smIncidenciaNme,smFasePersonalizado1Txt,smNodoId2,smNodoPersonalizado1Nbr,smNodoPersonalizado2Nbr,ssEnumFaseEstadoId,SmFaseUid,TraspasadoInd,JsonBodyTxt,JsonResponseTxt
154773,2025-05-02 05:51:10.350,AK02,ACABATS COURES 02,OF2503654,PE6XA3X0004,ALLONGE COUDEE 062ZF20024534BE2,45,AKS,0.0,0.0,0.0,0.0,0.0,0.0,1,NaN,NaN,NaN,AK02,NaN,NaN,FINALIZADA_OLANET,A664356A-0538-49CF-9101-B0859D7F8E93,1,"{""CodCompany"":""100"",""CodSite"":""01"",""Imputation...","{""Result"":true,""errorDetails"":[]}"


In [22]:
display(df_imputations.shape)
df_inicial = df_imputations.shape[0]

(196576, 26)

## Limpieza de atributos
Eliminar del dataset aquellos atributos prescindibles. 


In [23]:
def eliminar_columnas(df):
    """
    Elimina de df_imputations las columnas marcadas como innecesarias.
    Devuelve un DataFrame nuevo (no modifica el original).
    """

    cols_sobran = [
        "JsonResponseTxt", "JsonBodyTxt", "TraspasadoInd", "ssEnumFaseEstadoId", 
        "smNodoPersonalizado2Nbr", "smNodoPersonalizado1Nbr", "smNodoId2",
        "smFasePersonalizado1Txt", "CantidadManualOKNbr",
        "CantidadManualNOKNbr", "smFaseId",
        
    ]

    cols_a_quitar = [c for c in cols_sobran if c in columns_imputations]
    df_out = df.drop(columns=cols_a_quitar)

    # Aviso por si alguna de la lista no se encontró (typo o nombre distinto)
    no_encontradas = [c for c in cols_sobran if c not in df_imputations.columns]
    if no_encontradas:
        print(f"Aviso: {len(no_encontradas)} columnas de la lista no estaban en df: {no_encontradas}")

    print(f"Eliminadas {len(cols_a_quitar)} columnas. "
          f"Quedan {df_out.shape[1]} columnas, {df_out.shape[0]} filas.")
    return df_out

In [24]:
df_imputations_limpieza = eliminar_columnas(df_imputations)

Eliminadas 11 columnas. Quedan 15 columnas, 196576 filas.


In [25]:
display(df_imputations_limpieza.sample(1))

,FechaCreacionDte,smNodoId,smNodoNme,smOrdenId,smMaterialId,smOrdenNme,smFaseNme,MinutosRealNbr,MinutosPreparacionNbr,MinutosProduccionNbr,PorcentajeDedicacionNbr,ProductivoInd,smIncidenciaId,smIncidenciaNme,SmFaseUid
14935,2026-06-15 22:05:56.880,MA03,AUX.CON/TRT,OF2604778,BR2XA3X0247,COPPER PART Arm MB408060S,TRT,5.93333,0.0,5.93333,50.0,1,NaN,NaN,1ADB38DF-4F06-43A6-B0F3-CB1503DAE805


### Limpieza por intervalo de fechas
Trabajamos dentro del rango entre junio 2025 y antes de agosto 2026.

In [26]:
def filtrar_desde_fecha(df, fecha_desde, columna='FechaCreacionDte'):
    """Conserva solo las instancaias cuya fecha en 'columna' sea >= fecha_desde.
    Convierte la columna a datetime por seguridad (sirve si viene como texto).
    Devuelve un DataFrame nuevo."""

    n_inicial = df.shape[0]
    df_out = df.copy()

    # Convertir a datetime de forma robusta. errors='coerce' -> fechas no
    # parseables se vuelven NaT (nulo temporal) en vez de reventar.
    df_out[columna] = pd.to_datetime(df_out[columna], errors='coerce')

    # Avisar si alguna fecha no se pudo interpretar
    n_nat = df_out[columna].isna().sum()
    if n_nat:
        print(f"Aviso: {n_nat} filas con fecha no interpretable (NaT). "
              f"Quedan fuera del filtro >= {fecha_desde}.")

    limite = pd.to_datetime(fecha_desde)
    df_out = df_out[df_out[columna] >= limite].copy()

    print(f"Partida: {n_inicial} fases")
    print(f"Conservadas (desde {fecha_desde}): {df_out.shape[0]} "
          f"(descartadas {n_inicial - df_out.shape[0]})")
    return df_out

In [27]:
df_imputations_limpieza = filtrar_desde_fecha(df_imputations_limpieza, "2025-06-01")

Partida: 196576 fases
Conservadas (desde 2025-06-01): 141329 (descartadas 55247)


In [28]:
def filtrar_hasta_fecha(df, fecha_hasta, columna='FechaCreacionDte'):
    """Conserva solo las instancaias cuya fecha en 'columna' sea =< fecha_hasta.
    Convierte la columna a datetime por seguridad (sirve si viene como texto).
    Devuelve un DataFrame nuevo."""

    n_inicial = df.shape[0]
    df_out = df.copy()

    # Convertir a datetime de forma robusta. errors='coerce' -> fechas no
    # parseables se vuelven NaT (nulo temporal) en vez de reventar.
    df_out[columna] = pd.to_datetime(df_out[columna], errors='coerce')

    # Avisar si alguna fecha no se pudo interpretar
    n_nat = df_out[columna].isna().sum()
    if n_nat:
        print(f"Aviso: {n_nat} filas con fecha no interpretable (NaT). "
              f"Quedan fuera del filtro =< {fecha_hasta}.")

    limite = pd.to_datetime(fecha_hasta)
    df_out = df_out[df_out[columna] <= limite].copy()

    print(f"Partida: {n_inicial} fases")
    print(f"Conservadas (hasta {fecha_hasta}): {df_out.shape[0]} "
          f"(descartadas {n_inicial - df_out.shape[0]})")
    return df_out

In [29]:
df_imputations_limpieza = filtrar_hasta_fecha(df_imputations_limpieza, "2026-08-01")
df_intervalo_timepos = df_imputations_limpieza.shape[0] 

Partida: 141329 fases
Conservadas (hasta 2026-08-01): 140485 (descartadas 844)


Voy a guarda la tabla de las imputaciones en `res`

In [30]:
import pyarrow as pa

#df_imputations_limpieza["smFaseId"] = df_imputations_limpieza["smFaseId"].astype(str)

df_imputations_limpieza.to_parquet("res/procesed/df_imputations_limpieza.parquet")

# almaceno variables:
variables = {
    "Imputaciones iniciales": df_inicial,
    "Imputaciones dentro del intervalo temporal": df_intervalo_timepos
}
with open("res/procesed/variables/1.Tratamiento_Imputaciones.pkl", "wb") as f:
    pickle.dump(variables, f) 


In [31]:
tabla_resultados = pd.DataFrame({
    "Imputaciones iniciales": [df_inicial],
    "Imputaciones intervalo timepos": [df_intervalo_timepos]
}).T

tabla_resultados = tabla_resultados.rename(columns={0: "Número de imputaciones"})

print("=======================================================")
print("Resultados del notebook:")
print("=======================================================")
display(tabla_resultados)
print("=======================================================")

Resultados del notebook:


,Número de imputaciones
Imputaciones iniciales,196576
Imputaciones intervalo timepos,140485
